# Module 07: Interactive Modern File I/O & Safe Serialization

### What You Will Discover
By running this notebook, you will explore `pathlib.Path` cross-platform path handling, atomic file replacements to prevent data loss, and safe serialization across JSON and TOML.

**Key Question Answered:** *Why does writing directly to a destination file risk silent data corruption on power loss or process crashes?*


In [ ]:
# Step 1: pathlib.Path manipulation
from pathlib import Path

base_dir = Path('sample_sandbox')
nested_file = base_dir / 'config' / 'settings.json'
print(f'Composed Path: {nested_file}')
print(f'Parent directory: {nested_file.parent}')
print(f'Suffix: {nested_file.suffix}')


In [ ]:
# Step 2: Creating directories and writing safely
nested_file.parent.mkdir(parents=True, exist_ok=True)
nested_file.write_text('{"debug": true}', encoding='utf-8')
print(f'File exists: {nested_file.exists()}')
print(f'Content: {nested_file.read_text(encoding="utf-8")}')


In [ ]:
# Step 3: Cleanup sandbox
nested_file.unlink()
nested_file.parent.rmdir()
base_dir.rmdir()
print('Cleaned up sandbox directories.')


### 🔮 Prediction Prompt
**Before running the next cell:** In Python on Windows, if you run `open('test.txt', 'w').write('Rocket: 🚀')` without specifying `encoding='utf-8'`, what will happen when running under default Windows system locale `cp1252`?


In [ ]:
# Surprising Result: Encoding traps on Windows
test_emoji = 'Rocket: 🚀'
try:
    # Attempting to encode emoji as cp1252 fails!
    test_emoji.encode('cp1252')
    print('Encoded successfully.')
except UnicodeEncodeError as exc:
    print(f'Caught expected UnicodeEncodeError: {exc}')
    print('Explanation: Always declare encoding="utf-8" explicitly to prevent Windows crashes!')


### Atomic File Writes via `tempfile` and `os.replace`
Atomic writes guarantee that a reader never sees a half-written file if the writing process crashes mid-stream.


In [ ]:
import json
import os
import tempfile

target = Path('atomic_sample.json')
payload = {'service': 'auth', 'port': 8080, 'active': True}

with tempfile.NamedTemporaryFile('w', dir='.', delete=False, encoding='utf-8') as tf:
    temp_path = tf.name
    json.dump(payload, tf, indent=2)

os.replace(temp_path, target)  # Atomic on both POSIX and Windows
print(f'Successfully atomic-wrote {target.stat().st_size} bytes')
target.unlink()  # Cleanup


### Parsing Modern TOML with `tomllib` (Python 3.11+)
TOML is the standard configuration language for Python tools (PEP 518/621).


In [ ]:
import tomllib

toml_str = '''
[database]
host = "127.0.0.1"
port = 5432
enabled = true
'''
parsed = tomllib.loads(toml_str)
print(f'Parsed TOML dictionary: {parsed}')


### 🛠️ Interactive Challenge: Prevent Directory Traversal Attack
The following function accepts a filename from an untrusted client and attempts to read it from `upload_dir`. Vulnerability: an attacker passes `../../passwords.txt`. Fix the function to verify the resolved path remains inside `upload_dir`.


In [ ]:
# TODO: FIX ME - Enforce path validation to prevent directory traversal
def safe_read_upload(upload_dir: Path, user_filename: str) -> str:
    target = (upload_dir / user_filename).resolve()
    base = upload_dir.resolve()
    # FIX: Check target.is_relative_to(base)
    if not target.is_relative_to(base):
        raise PermissionError(f'Access Denied: Path traversal detected: {user_filename}')
    return 'File content safely verified'

try:
    safe_read_upload(Path('uploads'), '../../etc/shadow')
except (PermissionError, ValueError) as exc:
    print(f'Successfully blocked traversal attempt: {exc}')


### 🏁 Summary & Next Steps
- Use `pathlib.Path` for all path arithmetic.
- Always specify `encoding='utf-8'` in file I/O operations.
- Run `python 01_pathlib_and_file_io_demo.py` and `python 02_json_csv_toml_serialization_demo.py`.
- Complete the project in [PROJECT_GUIDE.md](PROJECT_GUIDE.md).
